In [1]:
%pip install bs4

Note: you may need to restart the kernel to use updated packages.


In [2]:
from mwtokenizer.tokenizer import Tokenizer
from bs4 import BeautifulSoup
import requests

def get_wiki_text(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    text_container = soup.find('div', {'id': 'mw-content-text'})
    
    # get all text from the paragraphs
    paragraphs = text_container.find_all('p')

    


In [3]:
def get_sections(text_nodes):
    merged_sections = []
    start = 0
    for node in text_nodes:
        text = node.get_text()
        previous_h2 = node.find_previous("div", {"class": "mw-heading"})
        section_title = "lead" if previous_h2 is None else previous_h2.find('h2').get_text()
        
        # If this is the first section or has a different title from the last merged section
        if not merged_sections or merged_sections[-1]['title'] != section_title:
            merged_sections.append({
                'title': section_title, 
                'text': text, 
                'start': start, 
                'end': start + len(text)
            })
        else:
            # Update the last section's text and end position
            last_section = merged_sections[-1]
            last_section['text'] += text
            last_section['end'] = start + len(text)
        
        start += len(text)
    
    return merged_sections

In [4]:
def get_sentences(sections):
    """
    Tokenize sentences from a list of sections with proper tracking of section, offset, and sentence metadata.
    
    :param sections: List of section dictionaries with 'title', 'text', and 'start' keys
    :return: List of sentence dictionaries with metadata
    """
    tokenizer = Tokenizer(language_code='en')
    sentences = []
    
    for section in sections:
        current_offset = section['start']
        
        for sen_id, sentence in enumerate(tokenizer.sentence_tokenize(section['text'], use_abbreviation=True)):
            sentences.append({
                'sen-id': sen_id,
                'section': section['title'],
                'start': current_offset,
                'end': current_offset + len(sentence),
                'sentence': sentence
            })
            current_offset += len(sentence)
    
    return sentences

In [5]:
import random
# make a method to score the sentences given a target entity
def score_sentences(sentences, target_entity):
    """
    Score sentences based on the presence of the target entity.
    
    :param sentences: List of sentence dictionaries with 'sentence' key
    :param target_entity: The target entity to score the sentences against
    :return: List of sentence dictionaries with 'score' key
    """
    for sentence in sentences:
        # for now give a random score
        sentence['score'] = random.random()
    return sentences

In [7]:
# # test the Bream article
# url = 'https://en.wikipedia.org/wiki/Bream'
# soup = BeautifulSoup(requests.get(url).text, 'html.parser')
# text_container = soup.find('div', id='mw-content-text').find_all(string=True)
# print(text_container)
# text = get_wiki_text(url)
# sentences = get_sentences(text)
# scored_sentences = score_sentences(sentences, 'Bream')

# for sentence in sentences:
#     print(sentence['sentence'])
#     print(sentence['section'])
#     print(sentence['start'])
#     print(sentence['end'])
#     print(sentence['sentence'])
#     print('---')
  
# for scored_sentence in scored_sentences:
#     print(scored_sentence['sentence'])
#     print(scored_sentence['score'])
#     print('---')


In [12]:
!pip3 install lxml

In [29]:
import requests
from lxml import html

url = 'https://en.wikipedia.org/wiki/Bream'
html_content = requests.get(url).text

# Parse the HTML content
tree = html.fromstring(html_content)

# XPath condition for text extraction
xpath_condition = """//div[@id='mw-content-text']//text()[
    not(ancestor-or-self::noscript) and
    not(ancestor-or-self::h2) and
    not(ancestor-or-self::h3) and
    not(ancestor-or-self::*/@class='mw-headline') and
    not(ancestor-or-self::*/@class='mw-headline mw-heading2') and
    not(ancestor-or-self::style) and 
    not(ancestor-or-self::cite) and
    not(ancestor-or-self::script) and
    not(ancestor-or-self::table) and
    not(ancestor-or-self::img) and
    not(ancestor-or-self::figure) and
    not(ancestor-or-self::sup) and
    not(ancestor-or-self::*/@class='mw-editsection') and
    not(ancestor-or-self::*/@class='mw-bandeau') and
    not(ancestor-or-self::*/@class='reflist') and
    not(ancestor-or-self::*/@class='map') and
    not(ancestor::div[
        contains(@class, "js-interprojects") or
        contains(@class, "galleryext") or
        contains(@class, "mw-headline") or
        contains(@class, "thumb") or 
        contains(@role, "navigation") or 
        contains(@class, "navbox") or 
        contains(@class, "toc") or 
        contains(@class, "reflist") or
        contains(@class, "printfooter") or
        contains(@class, "soslink") or
        contains(@style, "float:") or
        contains(@style, "display:none") or
        contains(@class, "bandeau") or
        contains(@class, "infobox") or
        contains(@class, "references") or
        contains(@class, "side-box") or
        contains(@class, "plainlinks") or
        contains(@class, "sisterbox") or
        contains(@class, "references")
    ])
]"""

# Extract text nodes
text_nodes = tree.xpath(xpath_condition)
#experiment with the text nodes
# Clean and print the extracted text
cleaned_text = ''.join(text for text in text_nodes)
print(cleaned_text)


This article is about a group of fish. For other uses, see Bream (disambiguation).
For the common bream of Europe, Abramis brama, see common bream.

Bream (/ˈbriːm/, US also /ˈbrɪm/ ) are species of freshwater fish belonging to a variety of genera including Abramis (e.g., A. brama, the common bream), Ballerus, Blicca, Brama, Chilotilapia, Etelis, Lepomis, Gymnocranius, Lethrinus, Nemipterus, Pharyngochromis, Rhabdosargus, Scolopsis, or Serranochromis.
Although species from all of these genera are called "bream", the term does not imply a degree of relatedness between them. Fish termed "bream" tend to be narrow, deep-bodied species. The name is a derivation of the  Middle English word breme, of Old French origin.



The term sea bream is sometimes used for fish of the family Sparidae including Acanthopagrus (Australia),  Argyrops, gilt-head bream Sparus aurata (orata in Italy, dorada in Spain), black seabream Spondyliosoma cantharus and red seabream Pagrus or Pagellus species; or pomfr

In [ ]:
import requests
from lxml import html

url = 'https://en.wikipedia.org/wiki/Bream'
html_content = requests.get(url).text

# Parse the HTML
tree = html.fromstring(html_content)

# -------------------------
# 1) Define XPaths
# -------------------------

# Text node extraction (to exclude certain tags/sections)
xpath_condition = """
//div[@id='mw-content-text']//text()[
    not(ancestor-or-self::noscript)
    and not(ancestor-or-self::h2)
    and not(ancestor-or-self::h3)
    and not(ancestor-or-self::*/@class='mw-headline')
    and not(ancestor-or-self::*/@class='mw-headline mw-heading2')
    and not(ancestor-or-self::style)
    and not(ancestor-or-self::cite)
    and not(ancestor-or-self::script)
    and not(ancestor-or-self::table)
    and not(ancestor-or-self::img)
    and not(ancestor-or-self::figure)
    and not(ancestor-or-self::sup)
    and not(ancestor-or-self::*/@class='mw-editsection')
    and not(ancestor-or-self::*/@class='mw-bandeau')
    and not(ancestor-or-self::*/@class='reflist')
    and not(ancestor-or-self::*/@class='map')
    and not(ancestor::div[
        contains(@class, "js-interprojects")
        or contains(@class, "galleryext")
        or contains(@class, "mw-headline")
        or contains(@class, "thumb") 
        or contains(@role, "navigation") 
        or contains(@class, "navbox") 
        or contains(@class, "toc") 
        or contains(@class, "reflist")
        or contains(@class, "printfooter")
        or contains(@class, "soslink")
        or contains(@style, "float:")
        or contains(@style, "display:none")
        or contains(@class, "bandeau")
        or contains(@class, "infobox")
        or contains(@class, "references")
        or contains(@class, "side-box")
        or contains(@class, "plainlinks")
        or contains(@class, "sisterbox")
        or contains(@class, "references")
    ])
]
"""

# H2 extraction (section headings)
xpath_section = """preceding::h2[
    ancestor::div[@id="mw-content-text"]
    and not(ancestor-or-self::noscript)
    and not(ancestor-or-self::style)
    and not(ancestor-or-self::cite)
    and not(ancestor-or-self::script)
    and not(ancestor-or-self::table)
    and not(ancestor::div[
        contains(@class, "thumb") 
        or contains(@role, "navigation") 
        or contains(@class, "navbox") 
        or contains(@class, "toc") 
        or contains(@class, "reflist")
        or contains(@class, "printfooter")
        or contains(@class, "soslink")
        or contains(@style, "float:")
    ])
][1]
"""

# -------------------------
# 2) Grab the nodes
# -------------------------
text_nodes = tree.xpath(xpath_condition)

# -------------------------
# 3) Build sections
# -------------------------
def build_sections(text_nodes):
    """
    For each text node, find the closest preceding <h2> in the DOM.
    Use its text content as the section title, or 'lead' if none.
    Accumulate text into sections with 'title', 'text', 'start', 'end'.
    """
    sections = []
    offset = 0  # tracks character offset across the entire text

    for node in text_nodes:
        # Clean up the extracted text
        text = node

        # Find the closest preceding h2 in the DOM
        # preceding::h2[1] means "the nearest h2 that appears before this node in document order"
        preceding_h2 = node.getparent().xpath(xpath_section)
        if preceding_h2:
            # Use its text_content as title
            section_title = preceding_h2[0].text_content().strip()
        else:
            # If there's no preceding <h2>, treat it as 'lead'
            section_title = "lead"

        # If this is the first section or a new title, start a new section
        if not sections or sections[-1]['title'] != section_title:
            sections.append({
                'title': section_title,
                'text': text,
                'start': offset,
                'end': offset + len(text)
            })
        else:
            # Otherwise, accumulate the text into the current (last) section
            sections[-1]['text'] += text
            sections[-1]['end'] = offset + len(text)

        # Move the global offset after adding this text
        offset += len(text)

    return sections

# Build the sections
sections = build_sections(text_nodes)

# -------------------------
# 4) Print or otherwise use your sections
# -------------------------
for sec in sections:
    print(sec['title'])
    print(sec['text'])
    print(sec['start'])
    print(sec['end'])
    print('---')

lead

This article is about a group of fish. For other uses, see Bream (disambiguation).
For the common bream of Europe, Abramis brama, see common bream.

Bream (/ˈbriːm/, US also /ˈbrɪm/ ) are species of freshwater fish belonging to a variety of genera including Abramis (e.g., A. brama, the common bream), Ballerus, Blicca, Brama, Chilotilapia, Etelis, Lepomis, Gymnocranius, Lethrinus, Nemipterus, Pharyngochromis, Rhabdosargus, Scolopsis, or Serranochromis.
Although species from all of these genera are called "bream", the term does not imply a degree of relatedness between them. Fish termed "bream" tend to be narrow, deep-bodied species. The name is a derivation of the  Middle English word breme, of Old French origin.




0
726
---
Marine species
The term sea bream is sometimes used for fish of the family Sparidae including Acanthopagrus (Australia),  Argyrops, gilt-head bream Sparus aurata (orata in Italy, dorada in Spain), black seabream Spondyliosoma cantharus and red seabream Pagru